# 03 — Task 2: target, history features and temporal splits
The interval-end convention is a declared user-accepted assumption. Retrospective archived observations are used; real-time performance is not claimed.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys
import zipfile

from IPython.display import display

BUCKET = os.environ.get("BUCKET", "sg-rainfall-dsa5208-261011-data")
def cloud_read(args):
    result = subprocess.run(["gcloud", *args], capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError("Cloud read failed; check the authorised project, object and VM permissions.")
    return result.stdout
runtime = json.loads(cloud_read(["storage", "cat", f"gs://{BUCKET}/config/runtime.json"]))
if runtime["billing_account_id"] != "01DAD8-AE5F1A-E3910A" or runtime["credit_id"] != "3LU63BPGY77JFDQQ":
    raise RuntimeError("Unexpected billing account or credit.")
PROJECT_ID = runtime["project_id"]
RUN_ID = runtime["run_id"]
ROOT = f"gs://{BUCKET}/runs/{RUN_ID}"
PROJECT_ROOT = Path.home() / "rainfall-project"
PROJECT_ROOT.mkdir(exist_ok=True)
bundle = PROJECT_ROOT / "source_bundle.zip"
cloud_read(["storage", "cp", f"gs://{BUCKET}/code/source_bundle.zip", str(bundle)])
if hashlib.sha256(bundle.read_bytes()).hexdigest() != runtime["source_bundle_sha256"]:
    raise RuntimeError("Source bundle hash differs from runtime manifest.")
with zipfile.ZipFile(bundle) as archive:
    for member in archive.namelist():
        if not (PROJECT_ROOT / member).resolve().is_relative_to(PROJECT_ROOT.resolve()):
            raise ValueError("Unsafe archive member.")
    archive.extractall(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(PROJECT_ROOT))
from rainfall.config import load_config, config_hash
config = load_config(PROJECT_ROOT / "config" / "experiment.json")
if config_hash(config) != runtime["config_sha256"]:
    raise RuntimeError("Experiment configuration differs from the uploaded manifest.")
billing = json.loads(cloud_read(["billing", "projects", "describe", PROJECT_ID, "--format=json"]))
if not billing.get("billingEnabled") or billing.get("billingAccountName") != "billingAccounts/01DAD8-AE5F1A-E3910A":
    raise RuntimeError("The project is not billed to Education.")
from pyspark.sql import SparkSession, functions as F
from pyspark import StorageLevel
from rainfall.runtime import spark_action, environment, require_distributed
spark = (SparkSession.builder.master("yarn").appName(f"rainfall-{RUN_ID}")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.executor.cores", "2")
    .config("spark.executor.instances", "4")
    .config("spark.dynamicAllocation.enabled", "false")
    .config("spark.sql.shuffle.partitions", str(config["shuffle_partitions"]))
    .getOrCreate())
spark.conf.set("spark.sql.session.timeZone", "Asia/Singapore")
spark.conf.set("spark.sql.ansi.enabled", "false")
spark.conf.set("rainfall.evidence.root", f"{ROOT}/evidence")
require_distributed(spark)
display(environment(spark))
display({"run_id": RUN_ID, "config_sha256": config_hash(config), "credit_id": runtime["credit_id"]})


In [ ]:
from rainfall.fixtures import run_fixtures
from rainfall.features import build_base_features, assign_splits, feature_dictionary
with spark_action(spark, RUN_ID, "target_and_leakage_fixtures"):
    display(run_fixtures(spark, config))
display(feature_dictionary(config))

In [ ]:
with spark_action(spark, RUN_ID, "base_features_and_labels"):
    clean = spark.read.parquet(f"{ROOT}/cleaned")
    base = assign_splits(build_base_features(clean, config), config)
    base.write.mode("errorifexists").partitionBy("split").parquet(f"{ROOT}/base_features")
    attrition = base.groupBy("split", "label_valid").agg(F.count("*").alias("base_samples"), F.min("epoch").alias("first_prediction_epoch"), F.max("epoch").alias("last_prediction_epoch"))
    attrition.write.mode("errorifexists").json(f"{ROOT}/sample_attrition")
    attrition.show(20, truncate=False)
    display({"decision_grid_policy": "observed exact-grid station/time keys; missing current observation times excluded from training/evaluation", "excluded_current_observation_slots": "documented by full-calendar coverage denominators", "history_policy": "partial windows retained with count/coverage indicators and train-fitted median imputation"})

In [ ]:
spark.stop()